# CUDA CNN kernels: test and benchmark on a free Colab GPU

1. **Runtime → Change runtime type → T4 GPU**, then run every cell in order.
2. Set `REPO` in the next cell to your GitHub repository.
3. The last cells write `results/bench_<gpu>.md`. Download it and commit it to the repo, then copy the numbers into the README.

In [ ]:
REPO = "https://github.com/Dhruvsanghavi07/cuda-cnn-kernels.git"  # <- change this
!nvidia-smi --query-gpu=name,driver_version,memory.total --format=csv
!nvcc --version | tail -2

In [ ]:
!rm -rf cuda-cnn-kernels && git clone -q $REPO cuda-cnn-kernels
%cd cuda-cnn-kernels
!cmake -S . -B build -DCMAKE_BUILD_TYPE=Release -DCMAKE_CUDA_ARCHITECTURES=native > /dev/null && cmake --build build -j 4 2>&1 | tail -3

## 1. Correctness: every kernel against the CPU reference

In [ ]:
!./build/test_reference
!./build/test_kernels

## 2. Benchmark: GEMM vs cuBLAS, convolution layers

In [ ]:
import subprocess, re
gpu = subprocess.run(["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"], capture_output=True, text=True).stdout.strip()
out_file = "results/bench_" + re.sub(r"[^A-Za-z0-9]+", "_", gpu).strip("_").lower() + ".md"
!mkdir -p results && ./build/bench --out {out_file}

## 3. Optional: the same convolution layers in PyTorch (cuDNN)

This is the bar to compare against: cuDNN picks the fastest of many hand-tuned algorithms per layer.

In [ ]:
import torch, torch.nn.functional as F
torch.backends.cudnn.benchmark = True
torch.backends.cudnn.allow_tf32 = False      # keep FP32 to match the custom kernels
torch.backends.cuda.matmul.allow_tf32 = False
layers = [("stem 7x7/2, 3->64, 256px", 1, 3, 256, 64, 7, 2, 3), ("enc1 3x3, 64->64, 64px", 1, 64, 64, 64, 3, 1, 1),
          ("enc2 3x3/2, 64->128, 64px", 1, 64, 64, 128, 3, 2, 1), ("enc3 3x3, 256->256, 16px", 1, 256, 16, 256, 3, 1, 1),
          ("enc4 3x3, 512->512, 8px", 1, 512, 8, 512, 3, 1, 1), ("dec 3x3, 64->64, 256px", 1, 64, 256, 64, 3, 1, 1),
          ("batch 8, 3x3, 64->64, 64px", 8, 64, 64, 64, 3, 1, 1)]
rows = ["| Layer | cuDNN ms |", "|---|---|"]
for name, n, c, hw, k, r, s, p in layers:
    x = torch.randn(n, c, hw, hw, device="cuda"); w = torch.randn(k, c, r, r, device="cuda"); b = torch.randn(k, device="cuda")
    f = lambda: F.relu(F.conv2d(x, w, b, stride=s, padding=p))
    for _ in range(5): f()
    torch.cuda.synchronize()
    times = []
    for _ in range(20):
        st, en = torch.cuda.Event(enable_timing=True), torch.cuda.Event(enable_timing=True)
        st.record(); f(); en.record(); torch.cuda.synchronize(); times.append(st.elapsed_time(en))
    rows.append(f"| {name} | {sorted(times)[10]:.3f} |")
table = "\n".join(rows)
print(table)
with open(out_file, "a") as fh:
    fh.write("\n## PyTorch / cuDNN reference (same layers, FP32, conv + bias + ReLU)\n\n" + table + "\n")

In [ ]:
from google.colab import files
print(open(out_file).read())
files.download(out_file)